# AuraSync AS Dataset —  Three-Class DS-CNN + INT8 Live Test


In [ ]:
# 1. Install/import dependencies and configure deterministic execution.
import sys, subprocess
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'tensorflow-hub>=0.16.1', 'librosa>=0.10.2', 'soundfile>=0.12.1',
    'scikit-learn>=1.3', 'pandas>=2.0', 'matplotlib>=3.7',
    'seaborn>=0.13', 'tqdm>=4.66'
])

import os, io, re, gc, json, math, time, base64, random, shutil, hashlib, zipfile, wave
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import tensorflow as tf
import tensorflow_hub as hub
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score
)
from IPython.display import display, Audio

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
print('Python:', sys.version.split()[0])
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


Python: 3.12.13
TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# 2. Project configuration.
PROJECT_NAME = 'AuraSync_AS_Final_3Class_HighAccuracy_INT8'
MODEL_VERSION = '2.0.0'
REPO_URL = 'https://github.com/GhimirePlan/AS_Final_Dataset_.git'
REPO_DIR = Path('/content/AS_Final_Dataset_')
WORK_DIR = Path('/content/aurasync_as_final_work')
DATASET_DIR = WORK_DIR / 'dataset_from_archives'
CACHE_DIR = WORK_DIR / 'cache'
EXPORT_DIR = WORK_DIR / 'export'

SEED = 20260713
SAMPLE_RATE = 16000
TARGET_SAMPLES = 16000
N_FFT = 512
WIN_LENGTH = 512
HOP_LENGTH = 320
N_MELS = 64
TIME_FRAMES = 49
FMIN = 80.0
FMAX = 7600.0
LOG_EPS = 1e-6

LABELS = ['stationary_background', 'speech_dominant', 'non_speech_activity']
LABEL_TO_ID = {name: i for i, name in enumerate(LABELS)}
ID_TO_LABEL = {i: name for name, i in LABEL_TO_ID.items()}
CLASS_COUNT = len(LABELS)

# Training strength and deployment acceptance gates.
STUDENT_SEEDS = [SEED, 1729]
EPOCHS = 70
BATCH_SIZE = 64
DISTILL_TEMPERATURE = 2.0
HARD_LOSS_WEIGHT = 0.78
DISTILL_LOSS_WEIGHT = 0.22 * (DISTILL_TEMPERATURE ** 2)
TARGET_TEST_ACCURACY = 0.85
TARGET_TEST_MACRO_F1 = 0.82
MAX_INT8_ACCURACY_DROP = 0.02

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

for directory in [WORK_DIR, CACHE_DIR, EXPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print('Labels:', LABEL_TO_ID)
print('Work directory:', WORK_DIR)


Labels: {'stationary_background': 0, 'speech_dominant': 1, 'non_speech_activity': 2}
Work directory: /content/aurasync_as_final_work
